In [1]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
from pathlib import Path
import sys
import time
from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import importlib
import pandas as pd

C:\Users\auder\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\auder\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
dossier_actuel = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
racine_projet = dossier_actuel.parent
sys.path.append(str(racine_projet))
%load_ext autoreload
%autoreload 2
import scripts.functions as fc

In [3]:
Path_data = racine_projet/"data"/"ECGData.mat"
Path_atome = racine_projet/"data"/"Atomes.mat"
X,y_classic,signaux,D = fc.Import_Data(Path_data,Path_atome)
_,y_chf,_,_=fc.Import_Data(Path_data,Path_atome,Sort='chf')
_,y_nsr1,_,_=fc.Import_Data(Path_data,Path_atome,Sort='nsr1')
_,y_nsr2,_,_=fc.Import_Data(Path_data,Path_atome,Sort='nsr2')

In [4]:
Models=[LogisticRegression(max_iter=10**9),tree.DecisionTreeClassifier(criterion='entropy'),RandomForestClassifier(),GaussianNB(),BernoulliNB(),
        KNeighborsClassifier(n_neighbors=10),KNeighborsClassifier(n_neighbors=11),KNeighborsClassifier(n_neighbors=12)]
ARR=96
CHF=ARR+30
Y=[y_classic,y_chf,y_nsr1,y_nsr2]
Sort=['classic','chf','nsr1','nsr2']

## Naive Classification :

In [28]:
importlib.reload(fc)

<module 'scripts.functions' from 'C:\\Users\\auder\\Documents\\Université\\L3\\Stage_ECG_git\\stage_ecg\\code\\scripts\\functions.py'>

In [9]:
for model in Models :
    tic = time.time()
    err,prtable=fc.Classifier_eq_iter(signaux,y_classic,model)
    print(f'ERR_{model}_Raw_100 :',err)
    print(prtable)
    print(f'time {model}_Raw_100 :',time.time()-tic)

ERR_LogisticRegression(max_iter=1000000000)_Raw_100 : 0.4190909090909091
   Precision     Recall   F1_Score
A  61.140007  77.336842  68.052409
C  50.248016  28.214286  33.760306
N  53.183622  33.732143  39.084705
time LogisticRegression(max_iter=1000000000)_Raw_100 : 408.5701570510864
ERR_DecisionTreeClassifier(criterion='entropy')_Raw_100 : 0.4200000000000001
   Precision     Recall   F1_Score
A  67.267522  70.184211  68.261014
C  31.911760  29.233333  29.273417
N  56.156183  50.089286  50.986390
time DecisionTreeClassifier(criterion='entropy')_Raw_100 : 2375.7177588939667
ERR_RandomForestClassifier()_Raw_100 : 0.28090909090909094
   Precision     Recall   F1_Score
A  68.065029  98.278947  80.354305
C  14.000000   2.366667   4.047619
N  95.499603  60.928571  72.850653
time RandomForestClassifier()_Raw_100 : 365.09498858451843
ERR_GaussianNB()_Raw_100 : 0.39454545454545453
   Precision     Recall   F1_Score
A  64.617880  76.094737  69.674265
C  54.233333  10.680952  17.335065
N  51.316

## Activation time classification

In [ ]:
for i in range(len(Y)) :# On classifie selon chacun des modes de tri
    print({Sort[i]})
    for model in Models :
        tic = time.time()
        err,prtable=fc.Classifier_eq_iter(X,Y[i],model,Sort=Sort[i])
        print(f'ERR_{model}_activ_100 :',err)
        print(prtable)
        print(f'time {model}_activ_100 :',time.time()-tic)

{'classic'}
ERR_LogisticRegression(max_iter=1000000000)_activ_100 : 0.24454545454545454
   Precision     Recall   F1_Score
A  71.689944  97.284211  82.489666
C  91.865079  86.695238  88.343717
N  66.833333  13.053571  21.271212
time LogisticRegression(max_iter=1000000000)_activ_100 : 1442.7374920845032
ERR_DecisionTreeClassifier(criterion='entropy')_activ_100 : 0.3442424242424241
   Precision     Recall   F1_Score
A  71.894777  75.157895  73.176930
C  56.127778  51.380952  51.324296
N  59.407712  53.125000  54.911811
time DecisionTreeClassifier(criterion='entropy')_activ_100 : 2269.472621202469
ERR_RandomForestClassifier()_activ_100 : 0.1733333333333333
   Precision     Recall   F1_Score
A  78.847931  97.055263  86.847132
C  98.916667  58.966667  71.958514
N  91.377381  65.285714  74.840882
time RandomForestClassifier()_activ_100 : 717.230742931366
ERR_GaussianNB()_activ_100 : 0.4109090909090908
   Precision      Recall   F1_Score
A    58.5201  100.000000  73.824131
C    31.0000    5.2

## Peak classification

In [ ]:
T=1/128
t=np.arange(0,len(signaux[0])*T,T)
t_activ_arr=[list(t[fc.find_peak(signal,0.7)]) for signal in signaux[:ARR]]
t_activ_chf=[list(t[fc.find_peak(signal,0.5)]) for signal in signaux[ARR:CHF]]
t_activ_nsr=[list(t[fc.find_peak(signal,2.1)]) for signal in signaux[CHF:]]
X_peak=[]
X_peak=t_activ_arr+t_activ_chf+t_activ_nsr
for x in X_peak :
    max=np.max([len(x) for x in X_peak])
    while len(x)<max :
        x.append(0)

In [ ]:
for i in range(len(Y)) :# On classifie selon chacun des modes de tri
    print({Sort[i]})
    for model in Models :# classification with the signals' peak
        tic = time.time()
        err,prtable=fc.Classifier_eq_iter(X_peak,Y[i],model,Sort[i])
        print(f'ERR_{model}_peak_100 :',err)
        print(prtable)
        print(f'time {model}_peak_100 :',time.time()-tic)

## Features Classification 

In [ ]:
features=fc.extract_features(signaux)
for i in range(len(Y)) :# On classifie selon chacun des modes de tri
    print({Sort[i]})
    for model in Models :# classification with the signals' features
        tic = time.time()
        err,prtable=fc.Classifier_eq_iter(features,Y[i],model,Sort[i])
        print(f'ERR_{model}_features_100 :',err)
        print(prtable)
        print(f'time {model}_features_100 :',time.time()-tic)

## Reconstruct signals Classification

In [ ]:
signaux_nsr_reconstruits = fc.reconstruire_signaux(z_nsr, d_nsr)
signaux_arr_reconstruits = fc.reconstruire_signaux(z_arr, d_arr)
signaux_chf_reconstruits = fc.reconstruire_signaux(z_chf, d_chf)
X_reconstruct=np.concatenate((signaux_arr_reconstruits,signaux_chf_reconstruits,signaux_nsr_reconstruits),axis=0)
for i in range(len(Y)) :# On classifie selon chacun des modes de tri
    print({Sort[i]})
    for model in Models :# classification with the signals' features
        tic = time.time()
        err,prtable=fc.Classifier_eq_iter(features,Y[i],model,Sort[i])
        print(f'ERR_{model}_reconstruct_100 :',err)
        print(prtable)
        print(f'time {model}_reconstruct_100 :',time.time()-tic)